In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
from torch import nn
from torch import optim
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torchvision.datasets import VOCSegmentation

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [2]:
# Preparing the dataset
import os

tar_filename = "VOCtrainval_25-May-2011.tar"
root_dir = "./VOCSegmentation/2011"
devkit_path = os.path.join(root_dir, "VOCdevkit")

if os.path.exists(devkit_path):
    print("The dataset is already prepared.")
else:
    print("Download the dataset...")
    !wget https://www.robots.ox.ac.uk/~vgg/projects/pascal/VOC/voc2011/VOCtrainval_25-May-2011.tar

    print("\nDownload complete. Extracting data...")
    !mkdir -p {root_dir}
    !tar -xf {tar_filename} -C {root_dir}/

    print(f"\nThe dataset is ready. ({devkit_path})")

Download the dataset...

Download complete. Extracting data...


'wget' �́A�����R�}���h�܂��͊O���R�}���h�A
����\�ȃv���O�����܂��̓o�b�` �t�@�C���Ƃ��ĔF������Ă��܂���B
�R�}���h�̍\��������Ă��܂��B



The dataset is ready. (./VOCSegmentation/2011\VOCdevkit)


tar: Error opening archive: Failed to open 'VOCtrainval_25-May-2011.tar'


In [ ]:
# Definition of FCN
class FCN(nn.Module):
    def __init__(self, backbone, num_classes=21):
        super(FCN, self).__init__()
        # backbone
        # 画像の特徴を抽出する部分
        # 既に学習済みのCNNをここに入れて使う
        self.backbone = backbone
        # convolution
        # self.FCNhead：nn.Sequentialで構成された変換器
        # nn.Cunv2d：Backboneが出力した特徴マップを128チャネルに圧縮している
        # nn.BatchNorm2d：学習を安定させるためのバッチ正規化
        # nn.ReLU()：非線形変換
        # nn.Dropout：過学習を防ぐためのドロップアウト
        # nn.Conv2d：最後にクラス分のチャネルに変換
        # ➡ これがピクセルごとの分類スコアになる
        self.FCNhead = nn.Sequential(nn.Conv2d(512, 128, 3, padding=1, bias=False),
                                      nn.BatchNorm2d(128),
                                      nn.ReLU(),
                                      nn.Dropout(0.1),
                                      nn.Conv2d(128, num_classes, 1))

    def forward(self, x):
        # 元の画像サイズを控えておく（後で元に戻すため）
        input_shape = x.shape[-2:]  # shape: (224, 224)
        # 画像をどんどんたたみこんで、意味情報をギュッと凝縮する
        x = self.backbone(x)  # WRITE ME # Processing in backbone (512, 7, 7)
        # 凝縮された特徴を、21クラスのスコアマップに変換する
        x = self.FCNhead(x)   # WRITE ME # Processing in FCNhead (21, 7 ,7)
        # F.interpolate()：小さくなったマップをbilonearで、元の（224, 224）サイズまで引き伸ばす
        # bilonear：画像や特徴マップを拡大する際に使われる計算方法で、まわりの4つの点から平均をとって
        # 新しい点の値を決める
        x = F.interpolate(    # WRITE ME  # upscale (21, 224, 224)
            x,
            size=input_shape,
            mode='bilinear',
            align_corners=False
            )
        return x

In [7]:
# セマンティックセグメンテーションの精度を評価するための指標である「mloU」を計算するためのクラス
# 混同行列を作って、loUを計算する
class mIoUScore(object):
    def __init__(self, n_classes):
        # 何種類のクラスを評価するのかを、インスタンスの中に保存
        self.n_classes = n_classes
        # 混同行列を初期化
        # ((n_classes, n_classes)：正解不正解のパターンを全てカウントするため
        self.confusion_matrix = np.zeros((n_classes, n_classes))

    def _fast_hist(self, label_true, label_pred, n_class):
        # 対象となるピクセルだけを抽出
        mask = (label_true >= 0) & (label_true < n_class)
        # np.bincount：ループを使わずに混同行列を計算
        # n_class * label_true[mask].astype(int) + label_pred[mask]
        # ➡ 正解クラスi、予測クラスjの組み合わせを「i × n_class + j」という
        # 1つの固有の数値に変換している
        hist = np.bincount(
            n_class * label_true[mask].astype(int) + label_pred[mask], minlength=n_class ** 2
        ).reshape(n_class, n_class) 
        # 混同行列を返す
        return hist

    # 推論結果をひたすら集計して、最終的な混同行列を慣性させる場所
    def update(self, label_trues, label_preds):
        # label_trues：バッチに入っている全画像の正解ラベル
        # label_preds：バッチに入っている全画像の予測ラベル
        # ➡ この二つをfor分でペアで取り出す
        for lt, lp in zip(label_trues, label_preds):
            # 1枚の画像から得られた「その画像の混同行列」をクラスが持っている
            # self.confusion_matrixに累積加算している
            self.confusion_matrix += self._fast_hist(lt.flatten(), lp.flatten(), self.n_classes)

    # 最終的なスコア「mloU」を算出する
    def get_scores(self):
        hist = self.confusion_matrix
        with np.errstate(divide='ignore', invalid='ignore'):
            # loUの計算部分
            # np.diag(hist)：行列の対角成分、つまり「予測も正解もクラスiだった数」
            # hist.sum(axis=1)：そのクラスが「正解」として現れた合計数
            # hist.sum(axis=0)：そのクラスが「予測」として現れた合計数
            # - np.diag(hist)：ダブって2回数えてしまった「正解した数」を1回分引く
            iou = np.diag(hist) / (hist.sum(axis=1) + hist.sum(axis=0) - np.diag(hist))
        # 各クラスごとのloUが計算出来たから、最後にその平均を取って「mloU」にする
        # np.nanmean：divide='ignore'で、もし分母が0だった時に「0で割ろうとしてる」警告が出ないようにするため
        mean_iou = np.nanmean(iou)
        return mean_iou

    # 今までの集計結果を全て消去して、リセットする
    def reset(self):
        self.confusion_matrix = np.zeros((self.n_classes, self.n_classes))